# Neural Network Foundations — Backpropagation from Scratch

**SENTINEL-CXR** — Uncertainty-Aware Chest Radiograph Triage
Deep Learning (MAIB AI 114) · Prof Anshul Gupta · S P Jain School of Global Management, Dubai

| Group member | Student ID |
|---|---|
| Krishna Mathur | AS25DXB018 |
| Atharva Soundankar | AS25DXB020 |
| Yash Petkar | AS25DXB021 |

---

**Syllabus mapping — Weeks 1–2: Introduction to Deep Learning; Neural Networks, Activation Functions, Backpropagation**

Before using a framework it is worth proving the gradient by hand. This notebook
implements a two-layer network with forward and backward passes in pure NumPy,
verifies the analytic gradient against a numerical one, then compares activation
functions and optimisers on the same problem.

The gradient check is the point: if the analytic and numerical gradients agree to
~1e-7, backpropagation is implemented correctly. Everything afterwards rests on it.


In [ ]:
# ── Environment ───────────────────────────────────────────────────────
# Runs on Colab free tier (T4). Nothing here needs a paid runtime.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install",
         "torchxrayvision", "scikit-learn", "seaborn"],
        check=False,
    )

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 20260812
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
})
INSTRUMENT, STAT = "#2E9CB8", "#D64541"

## 1. A two-layer network in NumPy

No autograd. Every derivative written out.

In [ ]:
class TwoLayerNet:
    """y = W2 @ act(W1 @ x + b1) + b2, trained by explicit backpropagation."""

    def __init__(self, n_in, n_hidden, n_out, activation="relu", seed=SEED):
        rng = np.random.default_rng(seed)
        # He initialisation for ReLU: variance 2/n_in keeps activation scale
        # stable through depth. Xavier (1/n_in) under-scales ReLU networks.
        self.W1 = rng.normal(0, np.sqrt(2.0 / n_in), (n_in, n_hidden))
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.normal(0, np.sqrt(2.0 / n_hidden), (n_hidden, n_out))
        self.b2 = np.zeros(n_out)
        self.activation = activation

    def _act(self, z):
        if self.activation == "relu":    return np.maximum(0, z)
        if self.activation == "tanh":    return np.tanh(z)
        if self.activation == "sigmoid": return 1 / (1 + np.exp(-np.clip(z, -50, 50)))
        if self.activation == "leaky":   return np.where(z > 0, z, 0.01 * z)
        raise ValueError(self.activation)

    def _act_grad(self, z):
        if self.activation == "relu":    return (z > 0).astype(float)
        if self.activation == "tanh":    return 1 - np.tanh(z) ** 2
        if self.activation == "sigmoid":
            s = self._act(z); return s * (1 - s)
        if self.activation == "leaky":   return np.where(z > 0, 1.0, 0.01)

    def forward(self, X):
        self.X  = X
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self._act(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        # Softmax with the max subtracted — exp of a large logit overflows.
        e = np.exp(self.z2 - self.z2.max(axis=1, keepdims=True))
        self.probs = e / e.sum(axis=1, keepdims=True)
        return self.probs

    def loss(self, y):
        n = y.shape[0]
        return -np.log(self.probs[np.arange(n), y] + 1e-12).mean()

    def backward(self, y):
        """Chain rule, written out. dL/dz2 for softmax+cross-entropy is (p - onehot)."""
        n = y.shape[0]
        dz2 = self.probs.copy()
        dz2[np.arange(n), y] -= 1
        dz2 /= n

        dW2 = self.a1.T @ dz2
        db2 = dz2.sum(axis=0)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * self._act_grad(self.z1)

        dW1 = self.X.T @ dz1
        db1 = dz1.sum(axis=0)
        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

print("TwoLayerNet defined")

## 2. Gradient check

The only way to know the derivation is right.

In [ ]:
def gradient_check(net, X, y, eps=1e-5):
    """Compare analytic gradients with central finite differences."""
    net.forward(X); grads = net.backward(y)
    report = {}
    for name in ["W1", "b1", "W2", "b2"]:
        param = getattr(net, name)
        numeric = np.zeros_like(param)
        it = np.nditer(param, flags=["multi_index"])
        while not it.finished:
            i = it.multi_index
            orig = param[i]
            param[i] = orig + eps; net.forward(X); lp = net.loss(y)
            param[i] = orig - eps; net.forward(X); lm = net.loss(y)
            param[i] = orig
            numeric[i] = (lp - lm) / (2 * eps)
            it.iternext()
        a, n_ = grads[name], numeric
        rel = np.abs(a - n_).max() / max(np.abs(a).max() + np.abs(n_).max(), 1e-12)
        report[name] = rel
    return report

rng = np.random.default_rng(SEED)
Xc, yc = rng.normal(size=(24, 8)), rng.integers(0, 3, 24)
net = TwoLayerNet(8, 12, 3)
for k, v in gradient_check(net, Xc, yc).items():
    verdict = "PASS" if v < 1e-6 else "FAIL"
    print(f"  {k:3s} relative error {v:.3e}   {verdict}")
print("\nBelow 1e-6 means backpropagation is analytically correct.")

## 3. Activation and optimiser ablation

Same data, same architecture, one variable at a time — this is what 'evaluate the performance of diverse models' means in practice.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X, y = make_moons(n_samples=2000, noise=0.25, random_state=SEED)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED)

def train_numpy(activation, lr=0.5, epochs=300, optimiser="sgd"):
    net = TwoLayerNet(2, 32, 2, activation=activation)
    state = {k: np.zeros_like(getattr(net, k)) for k in ["W1","b1","W2","b2"]}
    hist = []
    for ep in range(epochs):
        net.forward(Xtr); hist.append(net.loss(ytr)); g = net.backward(ytr)
        for k in state:
            if optimiser == "sgd":
                setattr(net, k, getattr(net, k) - lr * g[k])
            else:  # momentum
                state[k] = 0.9 * state[k] + g[k]
                setattr(net, k, getattr(net, k) - lr * state[k])
    acc = (net.forward(Xte).argmax(1) == yte).mean()
    return hist, acc

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
rows = []
for a in ["relu", "tanh", "sigmoid", "leaky"]:
    h, acc = train_numpy(a); rows.append((a, "sgd", acc))
    ax[0].plot(h, label=f"{a} ({acc:.3f})")
ax[0].set_title("Activation function"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(fontsize=7)

for o in ["sgd", "momentum"]:
    h, acc = train_numpy("relu", optimiser=o); rows.append(("relu", o, acc))
    ax[1].plot(h, label=f"{o} ({acc:.3f})")
ax[1].set_title("Optimiser"); ax[1].set_xlabel("epoch"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

print(pd.DataFrame(rows, columns=["activation","optimiser","test_accuracy"]).to_string(index=False))
print("\nSigmoid converges slowest: its gradient saturates toward 0 for |z| large,")
print("which is the vanishing-gradient problem that motivated ReLU.")

---

### References for this notebook

- Goodfellow, I., Bengio, Y. & Courville, A. (2016). *Deep Learning*, Ch. 6. MIT Press.
- LeCun, Y. et al. (1998). Gradient-based learning applied to document recognition. *Proc. IEEE*.
- Schmidhuber, J. (2015). Deep learning in neural networks: an overview. *Neural Networks*.

---

*SENTINEL-CXR is a student research prototype. It is not a medical device and
must not be used for clinical decisions.*
